In [1]:
import os
import re
from datetime import datetime

import pandas as pd
import numpy as np
import yfinance as yf
from tqdm import tqdm

In [2]:
# Configuration
START_DATE = "2001-03-15"
END_DATE = "2026-03-15"

print(f"Downloading daily data from {START_DATE} to {END_DATE}")

In [3]:
# Setup output directory
BASE = os.path.abspath(os.getcwd())
if os.path.basename(BASE) == "Data":
    BASE = os.path.dirname(BASE)

RAW_DIR = os.path.join(BASE, "Data", "Outputs", "Raw_Data")
os.makedirs(RAW_DIR, exist_ok=True)
print(f"Output directory: {RAW_DIR}")

Output directory: /Users/kamilkashif/Documents/University/Masters Thesis/Data_Fetch/Data/Outputs/Raw_Data


## Extract All Tickers from ndx_changes

In [4]:
def normalize_ticker(bloomberg_ticker):
    """Extract ticker symbol from Bloomberg format.
    
    Examples:
        'AAPL UW Equity' -> 'AAPL'
        'MSFT US Equity' -> 'MSFT'
        '9218611D UW Equity' -> '9218611D' (will fail on yfinance)
    """
    if pd.isna(bloomberg_ticker) or not bloomberg_ticker:
        return ""
    return str(bloomberg_ticker).strip().split()[0]


def extract_all_tickers_from_ndx_log(filepath):
    """Parse ndx_changes file and extract all unique ticker symbols."""
    if not os.path.isfile(filepath):
        raise FileNotFoundError(f"NDX log not found: {filepath}")
    
    tickers = set()
    with open(filepath, "r", encoding="utf-8", errors="replace") as f:
        in_add = False
        in_del = False
        
        for line in f:
            line = line.strip()
            if not line:
                continue
            
            # Detect section headers
            if "ADD" in line.upper() and "DEL" not in line.upper():
                in_add = True
                in_del = False
                continue
            if "DEL" in line.upper() and "ADD" not in line.upper():
                in_del = True
                in_add = False
                continue
            
            # Skip metadata lines
            if "TOTAL" in line.upper() or "PERIOD" in line.upper():
                continue
            if re.match(r"^-+$", line):  # separator lines
                continue
            if "!!!" in line or "[END" in line:
                continue
            
            # Extract tickers from ADD/DEL sections
            if in_add or in_del:
                # Skip words that are not tickers
                skip_words = ("ADD", "DEL", "PERIOD", "TOTAL", "Equity", "COUNT", 
                              "File", "Edit", "Format", "View", "Help", "LEND")
                
                # Split by common separators
                parts = re.split(r"[+*•\-]", line)
                for part in parts:
                    part = part.strip()
                    if not part:
                        continue
                    
                    # Extract first token (ticker with possible suffix)
                    tokens = part.split()
                    if not tokens:
                        continue
                    
                    ticker_with_suffix = tokens[0]
                    
                    # Skip if it's a skip word or pure number
                    if ticker_with_suffix in skip_words:
                        continue
                    if re.match(r"^\d+$", ticker_with_suffix):
                        continue
                    
                    # Normalize to get ticker symbol
                    ticker = normalize_ticker(ticker_with_suffix)
                    if ticker and re.match(r"^[A-Z0-9]{2,10}$", ticker.upper()):
                        tickers.add(ticker)
    
    return sorted(tickers)


# Find ndx_changes file
DATA_DIR = os.path.join(BASE, "Data")
NDX_LOG = os.path.join(DATA_DIR, "ndx_changes")
if not os.path.isfile(NDX_LOG):
    NDX_LOG = os.path.join(BASE, "ndx_changes")

# Extract all tickers
ALL_TICKERS = extract_all_tickers_from_ndx_log(NDX_LOG)
print(f"\nExtracted {len(ALL_TICKERS)} unique tickers from ndx_changes")
print(f"Sample: {ALL_TICKERS[:20]}")

# Identify Bloomberg-only IDs (numeric prefixes that won't work on yfinance)
bloomberg_only = [t for t in ALL_TICKERS if re.match(r"^\d{7,8}[A-Z]$", t)]
yfinance_compatible = [t for t in ALL_TICKERS if t not in bloomberg_only]

print(f"\nBloomberg-only IDs (will fail on yfinance): {len(bloomberg_only)}")
print(f"  {bloomberg_only}")
print(f"\nyfinance-compatible tickers: {len(yfinance_compatible)}")


Extracted 337 unique tickers from ndx_changes
Sample: ['0945329D', '0964591D', '1040983D', '1255459D', '1280712D', '1288453D', '1396924D', '1396926D', '1448062D', '1518855D', '1519128D', '1541931D', '1579957D', '166783Q', '1683351D', '1683997D', '1746513D', '1778808D', '1812212D', '1841189D']

Bloomberg-only IDs (will fail on yfinance): 44
  ['0945329D', '0964591D', '1040983D', '1255459D', '1280712D', '1288453D', '1396924D', '1396926D', '1448062D', '1518855D', '1519128D', '1541931D', '1579957D', '1683351D', '1683997D', '1746513D', '1778808D', '1812212D', '1841189D', '1918732D', '1920486D', '2207158D', '2217347D', '2293940D', '2293944D', '2297264D', '2297267D', '2307532Q', '2326248D', '2471693D', '2544554D', '2619871D', '3029830Q', '3122066Q', '3153670Q', '3437127Q', '3549162Q', '9210611D', '9990294D', '9996651D', '9999794D', '9999825D', '9999826D', '9999955D']

yfinance-compatible tickers: 293


## Download Data from yfinance

In [8]:
def download_ticker_data(ticker, start_date, end_date):
    """Download daily OHLCV data for a single ticker from yfinance.
    
    Returns:
        DataFrame with columns: date, open, high, low, close, volume
        None if download fails or data is empty
    """
    try:
        # Download data
        data = yf.download(
            ticker,
            start=start_date,
            end=end_date,
            interval="1d",
            progress=False
        )
        
        if data.empty:
            return None
        
        # Handle MultiIndex columns (yfinance returns MultiIndex even for single ticker)
        if isinstance(data.columns, pd.MultiIndex):
            # Flatten MultiIndex: ('Close', 'AAPL') -> 'Close'
            data.columns = data.columns.get_level_values(0)
        
        # Reset index to get date as a column
        data = data.reset_index()
        
        # Normalize column names to lowercase
        data.columns = [c.lower() for c in data.columns]
        
        # Select OHLCV columns (in correct order)
        required_cols = ['date', 'open', 'high', 'low', 'close', 'volume']
        available_cols = [c for c in required_cols if c in data.columns]
        
        if 'date' not in available_cols or 'close' not in available_cols:
            return None
        
        result = data[available_cols]
        
        # Ensure volume column exists (some tickers might not have it)
        if 'volume' not in result.columns:
            result['volume'] = 0
        
        return result
    
    except Exception as e:
        # Optionally print error for debugging
        # print(f"Error downloading {ticker}: {e}")
        return None

In [9]:
# Download data for all tickers
tickers_retrieved = []
tickers_not_retrieved = []
ticker_info = {}  # Track first/last date and row count

print(f"\nDownloading {len(ALL_TICKERS)} tickers...\n")

for ticker in tqdm(ALL_TICKERS, desc="Downloading"):
    df = download_ticker_data(ticker, START_DATE, END_DATE)
    
    if df is not None and len(df) > 0:
        # Save to CSV
        output_path = os.path.join(RAW_DIR, f"{ticker}.csv")
        df.to_csv(output_path, index=False)
        
        # Track info
        tickers_retrieved.append(ticker)
        ticker_info[ticker] = {
            'first_date': df['date'].min(),
            'last_date': df['date'].max(),
            'row_count': len(df)
        }
    else:
        tickers_not_retrieved.append(ticker)

print(f"\n{'='*60}")
print(f"DOWNLOAD SUMMARY")
print(f"{'='*60}")
print(f"Total tickers attempted: {len(ALL_TICKERS)}")
print(f"Successfully downloaded: {len(tickers_retrieved)}")
print(f"Failed/unavailable: {len(tickers_not_retrieved)}")
print(f"\nSuccess rate: {len(tickers_retrieved)/len(ALL_TICKERS)*100:.1f}%")

Downloading:   0%|          | 0/337 [00:00<?, ?it/s]HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: 0945329D"}}}
$0945329D: possibly delisted; no timezone found

1 Failed download:
['0945329D']: possibly delisted; no timezone found
Downloading:   0%|          | 1/337 [00:01<10:29,  1.87s/it]HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: 0964591D"}}}
$0964591D: possibly delisted; no timezone found

1 Failed download:
['0964591D']: possibly delisted; no timezone found
Downloading:   1%|          | 2/337 [00:03<10:22,  1.86s/it]$1040983D: possibly delisted; no timezone found

1 Failed download:
['1040983D']: possibly delisted; no timezone found
Downloading:   1%|          | 3/337 [00:04<07:27,  1.34s/it]$1255459D: possibly delisted; no timezone found

1 Failed download:
['1255459D']: possibly delisted; no timezone found
Downloading:   1%|          | 


DOWNLOAD SUMMARY
Total tickers attempted: 337
Successfully downloaded: 194
Failed/unavailable: 143

Success rate: 57.6%


In [10]:
# Show failed tickers breakdown
failed_bloomberg_ids = [t for t in tickers_not_retrieved if t in bloomberg_only]
failed_regular = [t for t in tickers_not_retrieved if t not in bloomberg_only]

print(f"\nFailed tickers breakdown:")
print(f"  Bloomberg IDs (expected to fail): {len(failed_bloomberg_ids)}")
print(f"    {failed_bloomberg_ids}")
print(f"\n  Regular tickers (delisted/unavailable): {len(failed_regular)}")
print(f"    {failed_regular}")


Failed tickers breakdown:
  Bloomberg IDs (expected to fail): 44
    ['0945329D', '0964591D', '1040983D', '1255459D', '1280712D', '1288453D', '1396924D', '1396926D', '1448062D', '1518855D', '1519128D', '1541931D', '1579957D', '1683351D', '1683997D', '1746513D', '1778808D', '1812212D', '1841189D', '1918732D', '1920486D', '2207158D', '2217347D', '2293940D', '2293944D', '2297264D', '2297267D', '2307532Q', '2326248D', '2471693D', '2544554D', '2619871D', '3029830Q', '3122066Q', '3153670Q', '3437127Q', '3549162Q', '9210611D', '9990294D', '9996651D', '9999794D', '9999825D', '9999826D', '9999955D']

  Regular tickers (delisted/unavailable): 99
    ['166783Q', '519239Q', '582663Q', '713075Q', 'ABGX', 'ADRX', 'ALXN', 'AMCC', 'ANSS', 'APOL', 'ARBA', 'ATHMQ', 'ATML', 'ATVI', 'ATYT', 'BBBYQ', 'BEAS', 'BGEN', 'BRCD', 'BRCM', 'BVSN', 'CDWC', 'CELG', 'CEPH', 'CERN', 'CHTRQ', 'CKFR', 'CMCSK', 'CTRX', 'CTXS', 'DISCA', 'DISCK', 'DISH', 'ENDPQ', 'ESRX', 'EXDSQ', 'FHCC', 'FLIR', 'FWLT', 'GENZ', 'GMCR', 'G

## Save Tracking Files

In [11]:
# Save tickers_retrieved.csv
df_retrieved = pd.DataFrame({
    'ticker': tickers_retrieved,
    'first_date': [ticker_info[t]['first_date'] for t in tickers_retrieved],
    'last_date': [ticker_info[t]['last_date'] for t in tickers_retrieved],
    'row_count': [ticker_info[t]['row_count'] for t in tickers_retrieved]
})
df_retrieved.to_csv(os.path.join(RAW_DIR, "tickers_retrieved.csv"), index=False)
print(f"Saved tickers_retrieved.csv ({len(tickers_retrieved)} tickers)")

# Save tickers_not_retrieved.csv
df_not_retrieved = pd.DataFrame({
    'ticker': tickers_not_retrieved,
    'reason': ['Bloomberg ID' if t in bloomberg_only else 'Unavailable/Delisted' 
               for t in tickers_not_retrieved]
})
df_not_retrieved.to_csv(os.path.join(RAW_DIR, "tickers_not_retrieved.csv"), index=False)
print(f"Saved tickers_not_retrieved.csv ({len(tickers_not_retrieved)} tickers)")

print(f"\nAll tracking files saved to: {RAW_DIR}")

Saved tickers_retrieved.csv (194 tickers)
Saved tickers_not_retrieved.csv (143 tickers)

All tracking files saved to: /Users/kamilkashif/Documents/University/Masters Thesis/Data_Fetch/Data/Outputs/Raw_Data


## Data Quality Check

In [12]:
# Show data coverage statistics
print("\nData Coverage Statistics:")
print(f"Earliest data: {df_retrieved['first_date'].min()}")
print(f"Latest data: {df_retrieved['last_date'].max()}")
print(f"\nRow count distribution:")
print(df_retrieved['row_count'].describe())

# Show tickers with limited data
limited_data = df_retrieved[df_retrieved['row_count'] < 1000].sort_values('row_count')
if len(limited_data) > 0:
    print(f"\nTickers with <1000 rows (recently listed or delisted):")
    print(limited_data[['ticker', 'first_date', 'last_date', 'row_count']].head(20))


Data Coverage Statistics:
Earliest data: 2001-03-15 00:00:00
Latest data: 2026-03-13 00:00:00

Row count distribution:
count     194.000000
mean     5058.561856
std      1827.223423
min       100.000000
25%      3930.750000
50%      6286.000000
75%      6286.000000
max      6286.000000
Name: row_count, dtype: float64

Tickers with <1000 rows (recently listed or delisted):
    ticker first_date  last_date  row_count
157   SOLS 2025-10-20 2026-03-13        100
130   PALM 2014-12-22 2020-01-08        239
79    GRAL 2024-06-12 2026-03-13        439
18     ARM 2023-09-14 2026-03-13        626
72    GEHC 2022-12-15 2026-03-13        812


## Sample Random Ticker

In [13]:
# Display a random ticker's data
if len(tickers_retrieved) > 0:
    random_ticker = tickers_retrieved[np.random.randint(0, len(tickers_retrieved))]
    df_sample = pd.read_csv(os.path.join(RAW_DIR, f"{random_ticker}.csv"))
    print(f"\nSample data for {random_ticker}:")
    print(f"Shape: {df_sample.shape}")
    print(f"\nFirst 5 rows:")
    print(df_sample.head())
    print(f"\nLast 5 rows:")
    print(df_sample.tail())
    print(f"\nColumn types:")
    print(df_sample.dtypes)


Sample data for TRI:
Shape: (5977, 6)

First 5 rows:
         date       open       high        low      close   volume
0  2002-06-12  16.448567  16.496014  16.132248  16.290407  6386423
1  2002-06-13  16.237691  16.443298  16.237691  16.443298   790045
2  2002-06-14  16.343122  16.343122  16.195507  16.337849   240654
3  2002-06-17  16.332584  16.453840  16.190241  16.448568   506345
4  2002-06-18  16.459109  16.459109  16.343126  16.422205   370948

Last 5 rows:
            date        open        high         low       close   volume
5972  2026-03-09  111.720001  113.349998  109.519997  111.519997  3083600
5973  2026-03-10  110.620003  111.519997  102.820000  103.699997  3492300
5974  2026-03-11  104.589996  106.449997  101.470001  103.110001  2400000
5975  2026-03-12  103.269997  106.169998   99.089996   99.279999  2135600
5976  2026-03-13   99.199997   99.720001   95.910004   96.339996  2719600

Column types:
date          str
open      float64
high      float64
low       float64